# Explore Whisper and RoBERTa for the SAILER reconstruction

This notebook inspects the exact foundation-model branches needed for the reconstruction:

- **Whisper** speech branch: `openai/whisper-large-v3` encoder hidden states. The first downstream model uses the last encoder layer.
- **RoBERTa** text branch: `roberta-large` hidden states. The downstream model uses a learned weighted average over encoder layers.

The notebook does not use dummy tensors, synthetic audio, small substitute models, or fake transcripts. If an actual audio file or transcript is unavailable, the corresponding forward pass is skipped or raises a clear error.


## 1. Imports and project paths

In [ ]:
from pathlib import Path
import os
import json
import warnings

import numpy as np
import pandas as pd
import torch
import torchaudio
import matplotlib.pyplot as plt

import transformers
from transformers import (
    AutoConfig,
    AutoModel,
    AutoTokenizer,
    WhisperModel,
    WhisperProcessor,
)

PROJECT_ROOT = Path.cwd().parent # Notebook has root from /notebooks always
assert PROJECT_ROOT.match("sailer_ser_reproduction")

DATA_DIR = PROJECT_ROOT / "data"
MANIFEST_DIR = DATA_DIR / "manifests" / "msp_podcast"
RAW_DIR = DATA_DIR / "raw" / "msp_podcast"
AUDIO_DIR = RAW_DIR / "Audio"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "foundation_model_exploration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MANIFEST_DIR:", MANIFEST_DIR)
print("AUDIO_DIR:", AUDIO_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. Version and device check

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

# Use fp16 only on CUDA. Keep CPU in fp32.
MODEL_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32
print("MODEL_DTYPE:", MODEL_DTYPE)

## 3. Exact model names and runtime switch

The model names below are the reconstruction targets. There are no small substitute models in this notebook.

Set `RUN_FORWARD_PASSES = True` only when the exact model weights are available locally or can be downloaded, and when your hardware has enough memory.


In [ ]:
WHISPER_MODEL_NAME = "openai/whisper-large-v3"
ROBERTA_MODEL_NAME = "roberta-large"

RUN_FORWARD_PASSES = True
MAX_AUDIO_SECONDS = 15.0
TARGET_SAMPLE_RATE = 16_000

print("Whisper model:", WHISPER_MODEL_NAME)
print("RoBERTa model:", ROBERTA_MODEL_NAME)
print("RUN_FORWARD_PASSES:", RUN_FORWARD_PASSES)
print("MAX_AUDIO_SECONDS:", MAX_AUDIO_SECONDS)


## 4. Load exact model configs

This cell loads configuration metadata for the exact models. It does not load full model weights.


In [ ]:
def load_config(model_name: str):
    try:
        return AutoConfig.from_pretrained(model_name)
    except Exception as exc:
        raise RuntimeError(
            f"Could not load config for {model_name!r}. "
            "Make sure the config is cached locally or internet access is available."
        ) from exc

whisper_cfg = load_config(WHISPER_MODEL_NAME)
roberta_cfg = load_config(ROBERTA_MODEL_NAME)

config_rows = [
    {
        "branch": "speech",
        "model": WHISPER_MODEL_NAME,
        "model_type": whisper_cfg.model_type,
        "num_encoder_layers": whisper_cfg.encoder_layers,
        "hidden_dim": whisper_cfg.d_model,
        "attention_heads": whisper_cfg.encoder_attention_heads,
        "max_source_positions": whisper_cfg.max_source_positions,
        "num_mel_bins": whisper_cfg.num_mel_bins,
    },
    {
        "branch": "text",
        "model": ROBERTA_MODEL_NAME,
        "model_type": roberta_cfg.model_type,
        "num_encoder_layers": roberta_cfg.num_hidden_layers,
        "hidden_dim": roberta_cfg.hidden_size,
        "attention_heads": roberta_cfg.num_attention_heads,
        "max_position_embeddings": roberta_cfg.max_position_embeddings,
        "vocab_size": roberta_cfg.vocab_size,
    },
]

config_df = pd.DataFrame(config_rows)
display(config_df)

config_out = OUTPUT_DIR / "foundation_model_configs.csv"
config_df.to_csv(config_out, index=False)
print("Saved:", config_out)


## 5. Expected hidden-state conventions

Hugging Face returns `hidden_states` as a tuple containing an initial embedding output followed by one tensor per encoder layer.

For our cache, use only the encoder layer outputs:

```python
layer_outputs = torch.stack(hidden_states[1:], dim=1)
```

Resulting conventions:

```text
whisper_hidden_states: [batch, whisper_encoder_layers, speech_frames, whisper_hidden_dim]
roberta_hidden_states: [batch, roberta_encoder_layers, text_tokens, roberta_hidden_dim]
text_attention_mask:   [batch, text_tokens]
```


In [ ]:
def stack_encoder_layer_outputs(hidden_states: tuple[torch.Tensor, ...]) -> torch.Tensor:
    """Convert HF hidden_states tuple to [B, L, T, D], excluding initial embeddings."""
    if len(hidden_states) < 2:
        raise ValueError("Expected hidden_states to contain embedding output plus encoder layer outputs.")
    return torch.stack(list(hidden_states[1:]), dim=1)


def masked_mean_pool(sequence: torch.Tensor, mask: torch.Tensor | None = None, eps: float = 1e-8) -> torch.Tensor:
    """Mean-pool [B, T, D] using an optional [B, T] mask."""
    if mask is None:
        return sequence.mean(dim=1)
    mask = mask.to(sequence.device).float()
    while mask.ndim < sequence.ndim:
        mask = mask.unsqueeze(-1)
    summed = (sequence * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp_min(eps)
    return summed / denom


def weighted_layer_average(layer_outputs: torch.Tensor, layer_logits: torch.Tensor) -> torch.Tensor:
    """Apply softmax-normalized layer weights to [B, L, T, D]."""
    if layer_outputs.ndim != 4:
        raise ValueError(f"Expected [B, L, T, D], got shape {tuple(layer_outputs.shape)}")
    if layer_logits.ndim != 1 or layer_logits.shape[0] != layer_outputs.shape[1]:
        raise ValueError(
            f"layer_logits must have shape [{layer_outputs.shape[1]}], got {tuple(layer_logits.shape)}"
        )
    weights = torch.softmax(layer_logits, dim=0)
    return (layer_outputs * weights.view(1, -1, 1, 1)).sum(dim=1)


## 6. Select an actual dataset sample

This cell selects an audio file from `data/manifests/msp_podcast/all.csv` if available, otherwise from `data/raw/msp_podcast/Audio/`. It does not create synthetic audio.


In [ ]:
def find_actual_audio() -> Path:
    all_csv = MANIFEST_DIR / "all.csv"
    if all_csv.exists():
        df = pd.read_csv(all_csv)
        if "audio_path" in df.columns:
            if "audio_exists" in df.columns:
                df = df[df["audio_exists"].astype(bool)]
            for path_str in df["audio_path"].dropna():
                path = Path(path_str)
                if not path.is_absolute():
                    path = PROJECT_ROOT / path
                if path.exists():
                    return path

    if AUDIO_DIR.exists():
        wavs = sorted(AUDIO_DIR.glob("*.wav"))
        if wavs:
            return wavs[0]

    raise FileNotFoundError(
        "No actual .wav file found. Expected either a valid audio_path in "
        f"{all_csv} or .wav files under {AUDIO_DIR}."
    )

sample_audio_path = find_actual_audio()
print("sample_audio_path:", sample_audio_path)


## 7. Load actual audio for Whisper

Whisper expects 16 kHz audio. This cell loads the selected waveform, converts to mono, resamples if needed, and truncates to the reconstruction limit.


In [ ]:
def load_audio_16khz(path: Path, max_seconds: float = 15.0) -> tuple[np.ndarray, int]:
    waveform, sr = torchaudio.load(str(path))  # [channels, samples]
    waveform = waveform.mean(dim=0)            # mono [samples]

    max_samples = int(max_seconds * sr)
    waveform = waveform[:max_samples]

    if sr != TARGET_SAMPLE_RATE:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=TARGET_SAMPLE_RATE)
        waveform = resampler(waveform)
        sr = TARGET_SAMPLE_RATE

    return waveform.numpy().astype(np.float32), sr

audio_np, sr = load_audio_16khz(sample_audio_path, max_seconds=MAX_AUDIO_SECONDS)
print("sample_rate:", sr)
print("audio shape:", audio_np.shape)
print("duration seconds:", round(audio_np.shape[0] / sr, 3))

plt.figure(figsize=(10, 2.5))
plt.plot(np.arange(len(audio_np)) / sr, audio_np)
plt.xlabel("time (seconds)")
plt.ylabel("amplitude")
plt.title(sample_audio_path.name)
plt.show()


## 8. Whisper forward pass and encoder hidden states

This extracts the exact speech-side tensors needed for later offline caching.


In [ ]:
if RUN_FORWARD_PASSES:
    whisper_processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_NAME)
    whisper_model = WhisperModel.from_pretrained(
        WHISPER_MODEL_NAME,
        torch_dtype=MODEL_DTYPE,
    ).to(DEVICE)
    whisper_model.eval()

    whisper_inputs = whisper_processor(
        audio_np,
        sampling_rate=sr,
        return_tensors="pt",
        return_attention_mask=True,
    )

    input_features = whisper_inputs.input_features.to(device=DEVICE, dtype=MODEL_DTYPE)

    print("input_features:", tuple(input_features.shape), input_features.dtype)
    print("model dtype:", next(whisper_model.parameters()).dtype)

    with torch.no_grad():
        whisper_encoder_outputs = whisper_model.encoder(
            input_features=input_features,
            output_hidden_states=True,
            return_dict=True,
        )

    whisper_layer_outputs = stack_encoder_layer_outputs(whisper_encoder_outputs.hidden_states)
    whisper_last_layer = whisper_layer_outputs[:, -1]
    whisper_pooled_last = masked_mean_pool(whisper_last_layer)

    print("returned hidden-state tensors:", len(whisper_encoder_outputs.hidden_states))
    print("whisper_layer_outputs:", tuple(whisper_layer_outputs.shape), whisper_layer_outputs.dtype)
    print("whisper_last_layer:   ", tuple(whisper_last_layer.shape), whisper_last_layer.dtype)
    print("whisper_pooled_last:  ", tuple(whisper_pooled_last.shape), whisper_pooled_last.dtype)
else:
    print("Skipped. Set RUN_FORWARD_PASSES = True after confirming the exact Whisper weights are available.")


## 9. Select an actual transcript for RoBERTa

RoBERTa needs transcript text. This cell looks for a transcript-like column in `all.csv`. If no transcript column exists, the RoBERTa forward pass cannot be run from the current manifests.

Accepted column names are:

```text
transcript, text, asr_text, sentence, utterance
```


In [ ]:
TRANSCRIPT_COLUMNS = ["transcript", "text", "asr_text", "sentence", "utterance"]


def find_actual_transcript_for_audio(audio_path: Path) -> str | None:
    all_csv = MANIFEST_DIR / "all.csv"
    if not all_csv.exists():
        return None

    df = pd.read_csv(all_csv)
    text_cols = [col for col in TRANSCRIPT_COLUMNS if col in df.columns]
    if not text_cols:
        return None

    # Prefer matching the selected audio filename.
    filename = audio_path.name
    if "filename" in df.columns:
        candidates = df[df["filename"] == filename]
    elif "audio_path" in df.columns:
        candidates = df[df["audio_path"].astype(str).map(lambda p: Path(p).name) == filename]
    else:
        candidates = df

    for col in text_cols:
        values = candidates[col].dropna().astype(str)
        values = values[values.str.strip().astype(bool)]
        if len(values) > 0:
            return values.iloc[0]

    # Fallback to any non-empty transcript in the manifest.
    for col in text_cols:
        values = df[col].dropna().astype(str)
        values = values[values.str.strip().astype(bool)]
        if len(values) > 0:
            return values.iloc[0]

    return None

sample_transcript = find_actual_transcript_for_audio(sample_audio_path)
print("sample_transcript:", sample_transcript)

if sample_transcript is None:
    print(
        "No actual transcript was found in the manifest. "
        "Add a transcript/asr_text/text column before running the RoBERTa forward pass."
    )


## 10. RoBERTa forward pass and encoder hidden states

This extracts the exact text-side tensors needed for later offline caching. It requires an actual transcript from the manifest.


In [ ]:
if RUN_FORWARD_PASSES and sample_transcript is not None:
    roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL_NAME)
    roberta_model = AutoModel.from_pretrained(
        ROBERTA_MODEL_NAME,
        torch_dtype=MODEL_DTYPE,
    ).to(DEVICE)
    roberta_model.eval()

    text_inputs = roberta_tokenizer(
        [sample_transcript],
        padding=True,
        truncation=True,
        return_tensors="pt",
    )
    text_inputs = {key: value.to(DEVICE) for key, value in text_inputs.items()}

    with torch.no_grad():
        roberta_outputs = roberta_model(
            **text_inputs,
            output_hidden_states=True,
            return_dict=True,
        )

    roberta_layer_outputs = stack_encoder_layer_outputs(roberta_outputs.hidden_states)

    # This is the same parameterization the downstream model will use.
    roberta_layer_logits = torch.zeros(
        roberta_layer_outputs.shape[1],
        dtype=roberta_layer_outputs.dtype,
        device=roberta_layer_outputs.device,
    )
    roberta_weighted_sequence = weighted_layer_average(roberta_layer_outputs, roberta_layer_logits)
    roberta_pooled = masked_mean_pool(roberta_weighted_sequence, text_inputs["attention_mask"])

    print("input_ids:              ", tuple(text_inputs["input_ids"].shape))
    print("attention_mask:         ", tuple(text_inputs["attention_mask"].shape))
    print("returned hidden states: ", len(roberta_outputs.hidden_states))
    print("roberta_layer_outputs:  ", tuple(roberta_layer_outputs.shape), roberta_layer_outputs.dtype)
    print("weighted sequence:      ", tuple(roberta_weighted_sequence.shape), roberta_weighted_sequence.dtype)
    print("pooled text embedding:  ", tuple(roberta_pooled.shape), roberta_pooled.dtype)
elif RUN_FORWARD_PASSES and sample_transcript is None:
    raise RuntimeError(
        "RUN_FORWARD_PASSES=True, but no actual transcript was found. "
        "Add a transcript/asr_text/text column to the manifest first."
    )
else:
    print("Skipped. Set RUN_FORWARD_PASSES = True after an actual transcript is available.")


## 11. Inspect module names for future freezing or LoRA experiments

For the current reconstruction, both foundation models are frozen and used offline for feature extraction. This inspection is included only to locate attention and feed-forward modules if LoRA is revisited later.


In [ ]:
def summarize_module_names(model: torch.nn.Module, max_rows: int = 80) -> pd.DataFrame:
    rows = []
    for name, module in model.named_modules():
        module_type = module.__class__.__name__
        is_candidate = any(token in name.lower() for token in [
            "q_proj", "k_proj", "v_proj", "out_proj", "query", "key", "value",
            "attention", "intermediate", "output"
        ])
        if is_candidate:
            rows.append({"name": name, "module_type": module_type})
    return pd.DataFrame(rows).head(max_rows)

if RUN_FORWARD_PASSES:
    if "whisper_model" in globals():
        print("Whisper candidate modules")
        display(summarize_module_names(whisper_model, max_rows=80))

    if "roberta_model" in globals():
        print("RoBERTa candidate modules")
        display(summarize_module_names(roberta_model, max_rows=80))
else:
    print("Skipped. This requires loaded exact model weights.")


## 12. Prototype cache record for the actual sample

This writes one cache file only if the exact forward passes have been run. The final extraction scripts should write this kind of record for every utterance under `data/processed/features/`.


In [ ]:
if RUN_FORWARD_PASSES and "whisper_layer_outputs" in globals():
    cache = {
        "utt_id": sample_audio_path.stem,
        "audio_path": str(sample_audio_path),
        "whisper_model_name": WHISPER_MODEL_NAME,
        "whisper_hidden_states": whisper_layer_outputs.squeeze(0).detach().cpu().half(),
    }

    if "roberta_layer_outputs" in globals():
        cache.update({
            "roberta_model_name": ROBERTA_MODEL_NAME,
            "roberta_hidden_states": roberta_layer_outputs.squeeze(0).detach().cpu().half(),
            "text_attention_mask": text_inputs["attention_mask"].squeeze(0).detach().cpu().bool(),
            "transcript": sample_transcript,
        })

    out_file = OUTPUT_DIR / f"{sample_audio_path.stem}_cached_hidden_states.pt"
    torch.save(cache, out_file)
    print("Saved:", out_file)
    print("File size MB:", round(out_file.stat().st_size / 1024**2, 2))

    loaded = torch.load(out_file, map_location="cpu")
    print("loaded keys:", sorted(loaded.keys()))
    print("whisper_hidden_states:", tuple(loaded["whisper_hidden_states"].shape), loaded["whisper_hidden_states"].dtype)
    if "roberta_hidden_states" in loaded:
        print("roberta_hidden_states:", tuple(loaded["roberta_hidden_states"].shape), loaded["roberta_hidden_states"].dtype)
else:
    print("Skipped. Run exact forward passes first.")


## 13. Practical conclusions for the reconstruction

- Feature-extraction scripts should call the exact frozen foundation models with `output_hidden_states=True`.
- Cache encoder layer outputs from `hidden_states[1:]`, not the initial embedding output.
- Save Whisper as all encoder layers, even though the first downstream speech model uses the last layer.
- Save RoBERTa as all encoder layers because the text model uses a learned weighted average across layers.
- Save cached tensors in `float16` on disk to reduce storage.
- Save text attention masks so token pooling ignores padding.
- The training dataloader should load cached tensors and labels; it should not rerun Whisper or RoBERTa every epoch.
